In [7]:
%load_ext autoreload
%autoreload 2

import os
import warnings
import pandas as pd
import numpy as np
from scipy import signal
from scipy.signal import savgol_filter
import math

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import seaborn as sns
%matplotlib qt

import tools.particle_swarm_optimization as pso
import tools.particle_swarm_optimization_plot as pso_plt
import tools.peristimulus_time_histogram as psth

warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
# read and proccees the lfp
ruta_principal = os.path.join(os.getcwd(), 'results/potjans_diesmann/')
carpetas = [nombre for nombre in os.listdir(ruta_principal)
            if os.path.isdir(os.path.join(ruta_principal, nombre))]

plt.close('all')

plot1, plot2, plot3 = True, True, True

aux=0
for carpeta in carpetas:
    folder = os.path.join(ruta_principal, carpeta)

    if aux == 1:
        continue
    aux=1    

    # read CSV into algo
    pdlfp = pd.read_csv(os.path.join(folder, 'lfp_layer_1.csv'), sep=' ', header=None, names=['lfp'])
    pdlfp2 = pd.read_csv(os.path.join(folder, 'lfp_layer_sph_1.csv'), sep=' ', header=None, names=['lfp'])
    
    lfp_signal = pdlfp['lfp'].to_numpy()
    fs = int(len(lfp_signal)/3)

    lfp_signal22 = pdlfp2['lfp'].to_numpy()
    fs2 = int(len(lfp_signal22)/3)

    # --- APLICA UN FILTRO PASO BAJO ---
    # Frecuencia de corte (e.g., 200 Hz)
    cutoff_freq = 200

    # Diseñar el filtro (Butterworth es una opción estándar)
    b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)

    # Aplicar el filtro a tu señal
    lfp_signal = signal.filtfilt(b, a, lfp_signal)

    cutoff_freq = 50
    # Diseñar el filtro (Butterworth es una opción estándar)
    b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)
    b2, a2 = signal.butter(4, cutoff_freq, btype='low', fs=fs2)

    # Aplicar el filtro a tu señal
    lfp_signal2 = signal.filtfilt(b, a, lfp_signal)
    lfp_signal22 = signal.filtfilt(b2, a2, lfp_signal22)

    time = np.arange(0, 3, 1/fs)
    time2 = np.arange(0, 3, 1/fs2)

    if plot1:
        fig, ax = plt.subplots(1, 1, layout='constrained', figsize=(10, 6), sharey=True)
        ax.plot(time, lfp_signal2)
        ax.plot(time2, lfp_signal22)
        ax.set_ylabel('LFP volt')
        ax.set_xlabel('Time (s)')
        ax.set_xlim(0, 3)
        y1, y2 = ax.get_ylim()

        ax.plot([1,1], [y1,y2],'--k')
        ax.plot([2,2], [y1,y2],'--k')
        ax.set_ylim(y1, y2)

        fig.suptitle('Optimización de Parámetros con PSO\nConectividad Lateral L2/3')
        plt.show()

    # epsctrum
    

    if plot2:
        time = np.arange(0, 3, 1/fs)

        len_win = int(len(lfp_signal)/3)
        
        for etapa in range (3):

            freqs, psd = signal.welch(lfp_signal[etapa*len_win:(etapa+1)*len_win], fs, nperseg=fs)
            print(freqs[0])
            print(freqs[-1])
            print(len(freqs))

            # Evitar log(0) añadiendo un valor muy pequeño si hay ceros en psd
            psd[psd == 0] = np.finfo(float).eps
            
            psd = psd[freqs <= 200]
            freqs = freqs[freqs <= 200]
        
            plt.figure(figsize=(10, 5))
            
            # 2. Usar plt.plot() normal, ya que los datos ya están en escala logarítmica
            plt.plot(freqs, psd)
            
            plt.title('Espectro de Potencia (LFP en dB)')
            plt.xlabel('Frecuencia (Hz)')
            
            # 3. Actualizar la etiqueta del eje Y
            plt.ylabel('Densidad Espectral de Potencia (dB/Hz)')
            
            #plt.grid(linestyle='--', alpha=0.7)
            plt.xlim(0, 200)
            plt.ylim(-50, 40)
            plt.show()


    # power spectrum
    f, t, Sxx = signal.spectrogram(lfp_signal, fs, nperseg=10000, nfft=20000)
    Sxx_db = 10 * np.log10(Sxx + np.finfo(float).eps)

    if plot3:

        plt.figure(figsize=(12, 6))
        plt.pcolormesh(t, f, Sxx_db, shading='gouraud', vmin=-25)#, vmax=35)
        plt.title('Espectrograma (LFP)')
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Frecuencia (Hz)')
        plt.colorbar(label='Potencia (dB)')
        
        # --- CAMBIO AQUÍ ---
        # Ajustamos el límite del eje Y para enfocarnos en el rango de LFP
        plt.ylim(0, 100)
        plt.plot([1, 1], [0, 100], '--k')
        plt.plot([2, 2], [0, 100], '--k')
        plt.show()



0.0
10000.0
10001
0.0
10000.0
10001
0.0
10000.0
10001


In [4]:
# read and proccees the lfp
ruta_principal = os.path.join(os.getcwd(), 'results/potjans_diesmann/')
carpetas = [nombre for nombre in os.listdir(ruta_principal)
            if os.path.isdir(os.path.join(ruta_principal, nombre))]

plt.close('all')

for carpeta in carpetas:
    folder = os.path.join(ruta_principal, carpeta) 

    for layer in range(4):
        # read CSV into algo
        pdlfp = pd.read_csv(os.path.join(folder, ('lfp_layer_'+str(layer+1)+'.csv')), sep=' ', header=None, names=['lfp'])

        pdlfp_sph = pd.read_csv(os.path.join(folder, ('lfp_layer_sph_'+str(layer+1)+'.csv')), sep=' ', header=None, names=['lfp'])
    
        lfp_signal = pdlfp['lfp'].to_numpy()
        lfp_signal_sph = pdlfp_sph['lfp'].to_numpy()

        fs = int(len(lfp_signal)/3)

        # Frecuencia de corte (e.g., 200 Hz)
        cutoff_freq = 200
        b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)

        # Aplicar el filtro a tu señal
        lfp_signal = signal.filtfilt(b, a, lfp_signal)
        lfp_signal_sph = signal.filtfilt(b, a, lfp_signal_sph)

        time = np.arange(0, 3, 1/fs)

        len_win = int(len(lfp_signal)/3)
        
        for etapa in range (3):

            freqs, psd_sph = signal.welch(lfp_signal_sph[etapa*len_win:(etapa+1)*len_win], fs, nperseg=fs)

            # Evitar log(0) añadiendo un valor muy pequeño si hay ceros en psd
            psd[psd == 0] = np.finfo(float).eps
            psd_sph[psd_sph == 0] = np.finfo(float).eps
            
            psd = psd[freqs <= 200]
            psd_sph = psd_sph[freqs <= 200]

            # --- CAMBIO AQUÍ ---
            # # 1. Convertir la potencia a decibeles
            # psd_db = 10 * np.log10(psd)
            # psd_db_sph = 10 * np.log10(psd_sph)
            #psd_db = psd
            
            lfp_csv = [layer+1, 0, etapa] + psd.tolist()
            df = pd.DataFrame([lfp_csv])
            df.to_csv(os.path.join(ruta_principal, 'lfp.csv'), mode='a', index=False, header=False)

            lfp_csv = [layer+1, 1, etapa] + psd_sph.tolist()
            df = pd.DataFrame([lfp_csv])
            df.to_csv(os.path.join(ruta_principal, 'lfp.csv'), mode='a', index=False, header=False)


NameError: name 'psd' is not defined

In [57]:
## plot mean psd
plt.close('all')

ruta_principal = os.path.join(os.getcwd(), 'results/potjans_diesmann/')

file_path = os.path.join(ruta_principal, 'lfp_100.csv')
df100 = pd.read_csv(file_path, header=None)
df100.rename(columns={
    0: 'layer',
    1: 'is_spherical',
    2: 'stage'
}, inplace=True)

file_path = os.path.join(ruta_principal, 'lfp_090.csv')
df90 = pd.read_csv(file_path, header=None)
df90.rename(columns={
    0: 'layer',
    1: 'is_spherical',
    2: 'stage'
}, inplace=True)

file_path = os.path.join(ruta_principal, 'lfp_075.csv')
df75 = pd.read_csv(file_path, header=None)
df75.rename(columns={
    0: 'layer',
    1: 'is_spherical',
    2: 'stage'
}, inplace=True)

file_path = os.path.join(ruta_principal, 'lfp_050.csv')
df50 = pd.read_csv(file_path, header=None)
df50.rename(columns={
    0: 'layer',
    1: 'is_spherical',
    2: 'stage'
}, inplace=True)

color = ['#213BFF','#9E0FFF','#FF0052','#FF7600']

labels = ['Control', 'Inh at 90%', 'Inh at 75%', 'Inh at 50%']
lays = ['L2/3']

esf = 1
lay = 1

fig, ax = plt.subplots(1, 3, layout='constrained', figsize=(13, 5), sharey=True)

for etapa in range(3):


    # Define los valores que quieres buscar
    layer_buscado = lay+1
    esferica_buscada = esf  # 1 para esférica, 0 para no esférica
    etapa_buscada = etapa

    # Crear una máscara booleana combinando las condiciones con '&' (y)
    # Es importante usar paréntesis para cada condición
    mascara = (df90['layer'] == layer_buscado) & \
            (df90['is_spherical'] == esferica_buscada) & \
            (df90['stage'] == etapa_buscada)

    # Aplicar la máscara al DataFrame para obtener solo las filas que coinciden
    resultados100 = df100[mascara].values
    resultados90 = df90[mascara].values
    resultados75 = df75[mascara].values
    resultados50 = df50[mascara].values
    resultados100 = 10 * np.log10(resultados100.astype(float))
    resultados90 = 10 * np.log10(resultados90.astype(float))
    resultados75 = 10 * np.log10(resultados75.astype(float))
    resultados50 = 10 * np.log10(resultados50.astype(float))    
    # resultados100 = resultados100.astype(float)
    # resultados90 = resultados90.astype(float)
    # resultados75 = resultados75.astype(float)
    # resultados50 = resultados50.astype(float)

    resultados = np.stack((resultados100, resultados90, resultados75, resultados50), axis=1)
    freqs = np.linspace(0.0, 200.0, 204)

    mean_cnd = np.mean(resultados,axis=0)
    std_cnd = np.std(resultados,axis=0)

    # --- Parámetros del filtro ---
    window_length = 13  # Debe ser un número impar
    polyorder = 3       # Orden del polinomio (ej. 2 para cuadrático, 3 para cúbico)

    # Suavizar el vector 1D
    mean_cnd = savgol_filter(mean_cnd, window_length, polyorder)
    std_cnd = savgol_filter(std_cnd, window_length, polyorder, axis=1)

    # coeficientes = np.polyfit(log_freqs, log_psd, 1)
    # pendiente = coeficientes[0]
    # intercepto = coeficientes[1]

    for i in range(4):

        ax[etapa].plot(freqs[2:], mean_cnd[i,2:], lw=2, c=color[i], 
                label=labels[i])
        ax[etapa].fill_between(freqs[2:], (mean_cnd[i,2:]-std_cnd[i,2:]/1.5), (mean_cnd[i,2:]+std_cnd[i,2:]/1.5), color=color[i], alpha=.1)

    ax[etapa].tick_params(axis='both', labelsize=15)

    for spine in ax[etapa].spines.values():
        spine.set_linewidth(2)

    if etapa==0:
        ax[etapa].set_ylabel('PSD [db]', fontsize=15)

    if etapa<=2:
        ax[etapa].legend(loc='upper right', fancybox=True, shadow=False, ncol=1)

    texto = ['Basal activity','Classical receptive field','Extra-classical receptive field']
    ax[etapa].set_xlabel('Freq [hz]', fontsize=15)
    #ax[etapa].set_xlim([2,100])
    ax[etapa].set_title(texto[etapa], fontsize=15)

    plt.suptitle(f'PSD under varying inhibition levels (All layers lateral connectivity)', fontsize=22)
    #plt.suptitle(f'PSD under varying inhibition levels (L2/3 lateral connectivity)', fontsize=22)


qt.qpa.xcb: QXcbConnection: XCB error: 3 (BadWindow), sequence: 33875, resource id: 9232872, major code: 40 (TranslateCoords), minor code: 0
